In [14]:
import operator
from typing import Annotated, TypedDict
from datetime import datetime, timedelta
from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# 1. Tools: Location, Date, and Weather
@tool
def get_current_location():
    """Returns the user's current city based on system IP/GPS."""
    return "Hong Kong"

@tool
def get_current_date():
    """Returns today's date."""
    return f"Today is {datetime.now().strftime('%Y-%m-%d')}."

@tool
def get_historical_weather(location: str, date: str):
    """Fetches weather for a specific city and date."""
    return f"The weather in {location} on {date} was Humid, 22°C."

tools = [get_current_location, get_current_date, get_historical_weather]

# 2. Model Setup
model = ChatOllama(model="qwen2.5:7b", temperature=0).bind_tools(tools)

# 3. Graph Construction
class State(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

def call_model(state: State):
    return {"messages": [model.invoke(state["messages"])]}

workflow = StateGraph(State)
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(tools))

workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", lambda x: "tools" if x["messages"][-1].tool_calls else END)
workflow.add_edge("tools", "agent")

# 4. Compile & Run (Fully Automated)
app = workflow.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "auto_location_weather"}}
# Updated Question: Asking for location and relative time weather
inputs = {"messages": [HumanMessage(content="我現在在哪個城市？查今天的日期，然後告訴我那個城市昨天的天氣如何")]}

for event in app.stream(inputs, config, stream_mode="values"):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

我現在在哪個城市？查今天的日期，然後告訴我那個城市昨天的天氣如何
================================== Ai Message ==================================

为了回答您的问题，我将首先获取您当前所在的城市。然后，我会查询今天的日期，并根据这些信息告诉您昨天该城市的天气情况。
Tool Calls:
  get_current_location (452c72de-e994-414d-bb26-c4ff4df25e9a)
 Call ID: 452c72de-e994-414d-bb26-c4ff4df25e9a
  Args:
================================= Tool Message =================================
Name: get_current_location

Hong Kong
================================== Ai Message ==================================
Tool Calls:
  get_current_date (9caad46c-8f85-491f-860f-d6d06b0b7c34)
 Call ID: 9caad46c-8f85-491f-860f-d6d06b0b7c34
  Args:
================================= Tool Message =================================
Name: get_current_date

Today is 2026-03-29.
================================== Ai Message ==================================

您当前所在的城市是香港。今天是2026年3月29日，根据这个信息，昨天的日期是2026年3月28日。现在我将查询2026年3月28日香港的天气情况。
Tool Ca